# MoveScope 模板敏感性实验

评估用于构建模板的专家特征序列数量如何影响评分稳定性。

In [ ]:
from pathlib import Path
import sys

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "movescope").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from movescope.experiments import load_feature_sequences, run_template_sensitivity_from_features

FIG_DIR = ROOT / "docs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
expert_dir = ROOT / "data" / "features" / "expert_squat"
test_dir = ROOT / "data" / "features" / "test_squat"

expert_sequences = load_feature_sequences(expert_dir)
test_sequences = load_feature_sequences(test_dir)

print(f"专家序列：{len(expert_sequences)} 条，来源：{expert_dir}")
print(f"测试序列：{len(test_sequences)} 条，来源：{test_dir}")


In [ ]:
rows = run_template_sensitivity_from_features(
    expert_sequences,
    test_sequences,
    counts=(1, 3, 5, 10),
)

df = pd.DataFrame(rows)
if df.empty:
    print("没有可用的敏感性实验记录，请向 data/features/expert_squat 和 data/features/test_squat 添加 .npy/.npz 特征数组。")
else:
    display(df)


In [ ]:
if not df.empty:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(df["template_count"], df["std_score"], marker="o", color="#b94331", linewidth=2)
    ax.set_title("专家模板数量与评分稳定性")
    ax.set_xlabel("专家特征序列数量")
    ax.set_ylabel("评分标准差")
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    output = FIG_DIR / "template_sensitivity.png"
    fig.savefig(output, dpi=160)
    print(f"图表已保存：{output}")


In [ ]:
if not df.empty:
    best = df.sort_values("std_score").iloc[0]
    print(
        f"使用 {int(best.template_count)} 条专家序列时评分标准差最低，为 {best.std_score:.2f}，"
        f"共评估 {int(best.n_tests)} 条测试序列。"
    )
